In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

# add project root (parent of current file) to path
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
             
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scripts.TPS import ThinPlateSpline
from scripts.plotting import *

# ---------- Load vector field ----------
def load_vector_field(csv_path):
    df = pd.read_csv(csv_path)
    X = df[["x", "y"]].values
    V = df[["vx", "vy"]].values
    time = df["time"].values
    return X, V, time

# ---------- File organization ----------
path_map = {
    "straight_line": "./data/1d/straight_line.csv",
    "sine_curve": "./data/1d/sine_curve.csv",
    "branch_2": "./data/1d/branch_2.csv",
    "branch_4": "./data/1d/branch_4.csv",
    "rotation": "./data/2d/rotation.csv",
    "spiral": "./data/2d/spiral.csv",
    "saddle": "./data/2d/saddle.csv",
    "quadratic_source_sink": "./data/2d/quadratic_source_sink.csv"
}

# Custom layout
row1_names = ["straight_line", "sine_curve", "branch_2", "branch_4"]
row2_names = ["rotation", "spiral", "saddle", "quadratic_source_sink"]
plot_order = row1_names + row2_names

fig, axs = plt.subplots(1, 8, figsize=(32, 4))

for ax, name in zip(axs, plot_order):
    X, V, time = load_vector_field(path_map[name])

    # normalize color to [0,1] for consistent colormap
    time = (time - time.min()) / (time.max() - time.min() + 1e-12)

    # Fit TPS *only* to ground-truth vector field
    tps_vf = ThinPlateSpline(X, n_control_points=100)
    tps_vf.fit(V, dof=15)

    # ---- panel-specific layout ----
    if name == "straight_line":
        stream_density, aspect = 0.2, 2.5
    elif name == "sine_curve":
        stream_density, aspect = 0.6, 2.0
    elif name == "branch_2":
        stream_density, aspect = 0.6, 1.5
    elif name == "branch_4":
        stream_density, aspect = 0.8, 1.5
    else:
        stream_density, aspect = 0.6, "equal"

    # ---- plot ----
    plot_velocity_streamplot(
        X_2d=X,
        tps_vf=tps_vf,
        grid_density=1.0,
        stream_density=stream_density,
        scatter_color=time,
        scatter_size=400,
        scatter_alpha=0.2,
        ax=ax,
        # title=name.replace("_", " "),
        aspect=aspect,
        vmin=0.0,
        vmax=1.0,
        arrowsize=3.0,
        cmap="viridis",
        show_axes=False,
        streamline_thickness=4.0,
        grid_size=50
    )

plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ------------------------------------------
# simulate noisy data (your block unchanged)
# ------------------------------------------
simulation_results = {}
noise, extra_dim = 0.2, 6
for name, path in path_map.items():
    X_gt, V_gt, time = load_vector_field(path)
    X_noisy = X_gt + np.random.normal(scale=noise, size=X_gt.shape)
    V_noisy = V_gt + np.random.normal(scale=noise, size=V_gt.shape)
    X_dummy = np.random.normal(scale=noise, size=(X_gt.shape[0], extra_dim))
    V_dummy = np.random.normal(scale=noise, size=(V_gt.shape[0], extra_dim))
    X = np.hstack([X_noisy, X_dummy])
    V = np.hstack([V_noisy, V_dummy])
    simulation_results[name] = dict(X=X, V=V, X_gt=X_gt, V_gt=V_gt, true_time=time)

In [ ]:
from scripts.VectorFieldEmbedder import *

embedding_results = {}

for name, data in simulation_results.items():
    X = data["X"]
    V = data["V"]
    time = data["true_time"]
    time = (time - np.min(time)) / (np.max(time) - np.min(time))  # normalize

    emb = VectorFieldEmbedder(
        X, V, use_PCA=False,
        embed_kwargs={"n_neighbors": 20, "min_dist": 0.4}
    )
    emb.initialize_embedding()
    # emb.optimize()

    embedding_results[name] = {
        "embedder": emb,
        "time": time
    }

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os
from scripts.evaluation import evaluate_embedding_method

fig, axs = plt.subplots(1, 8, figsize=(32, 4))
scores = []

for ax, name in zip(axs, embedding_results.keys()):
    emb = embedding_results[name]["embedder"]
    time = embedding_results[name]["time"]
    X_gt = simulation_results[name]["X_gt"]
    V_gt = simulation_results[name]["V_gt"]
    X_emb = emb.X_emb
    V_emb = emb.tps_vf.predict(X_emb)

    # Custom panel layout
    if name == "straight_line":
        stream_density, aspect = 0.4, 2.5
    elif name == "sine_curve":
        stream_density, aspect = 0.6, 2.0
    elif name == "branch_2":
        stream_density, aspect = 0.6, 1.5
    elif name == "branch_4":
        stream_density, aspect = 0.8, 1.5
    else:
        stream_density, aspect = 0.6, "equal"

    # Plot
    plot_velocity_streamplot(
        X_2d=X_emb,
        tps_vf=emb.tps_vf,
        grid_density=1.1, 
        stream_density=stream_density,
        scatter_color=time,
        scatter_size=400,
        scatter_alpha=0.2,
        ax=ax,
        title=None,
        aspect=aspect,
        vmin=0.0,
        vmax=1.0,
        arrowsize=3.0,
        cmap="viridis",
        show_axes=False,
        streamline_thickness=4.0,
        grid_size=50,
        pad_frac=0.1
    )

    # Score
    metrics = evaluate_embedding_method(X_gt, X_emb, V_gt, V_emb, k=30)
    metrics["dataset"] = name
    scores.append(metrics)

plt.tight_layout()
fig_dir = "./figures/simulation"
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, "flowmap_embedding_streams.png")
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
print(f"Saved figure to {fig_path}")
plt.show()

# Save scores
df = pd.DataFrame(scores).set_index("dataset")
os.makedirs("./data/8_vf_collection", exist_ok=True)
df.to_csv("./data/8_vf_collection/flowmap.csv")

print("\nFlowMap scores saved to ./data/8_vf_collection/flowmap.csv")
print(df.round(4))